# 📸 Snap.py Backup Tool - Google Colab Edition (Lite)
Zoptymalizowana wersja do tworzenia backupów stron www.

**Instrukcja:**
1. Uruchom komórkę **INSTALACJA** (może potrwać 2-3 minuty).
2. Uruchom komórkę **PODŁĄCZ DRIVE**.
3. Edytuj listę stron w komórce **KONFIGURACJA**.
4. Uruchom **START BACKUPU**.

In [ ]:
# --- KOMÓRKA 1: INSTALACJA ZALEŻNOŚCI ---
# Instalujemy Chrome, Chromedriver i biblioteki Python
!echo "Rozpoczynanie instalacji..."
!apt-get update -qq
!apt-get install -y -qq chromium-chromedriver libusb-1.0-0-dev wget
!pip install -q selenium webdriver-manager requests beautifulsoup4 lxml Pillow
!echo "Instalacja zakończona. Sprawdź czy nie ma czerwonych błędów powyżej."

In [ ]:
# --- KOMÓRKA 2: PODŁĄCZ GOOGLE DRIVE ---
from google.colab import drive
import os

print("Montowanie Dysku Google...")
drive.mount('/content/drive')

# Ścieżka zapisu
BACKUP_DIR = "/content/drive/MyDrive/Web_Backups"
if not os.path.exists(BACKUP_DIR):
    os.makedirs(BACKUP_DIR)
    print(f"Utworzono folder backupów: {BACKUP_DIR}")
else:
    print(f"Folder backupów istnieje: {BACKUP_DIR}")

In [ ]:
# --- KOMÓRKA 3: KONFIGURACJA I LISTA STRON ---
# Edytuj tę listę według potrzeb
URL_LIST = [
    "https://www.google.com",
    "https://www.wikipedia.org",
    # Dodaj swoje strony tutaj
]

# Tryb pracy: 'full', 'screenshots', 'clean'
MODE = 'full'

# Ustawienia przeglądarki
VIEWPORT_WIDTH = 1920
VIEWPORT_HEIGHT = 1080
WAIT_TIME = 3  # Sekundy czekania na załadowanie JS

print(f"Konfiguracja zatwierdzona. Tryb: {MODE}, Stron do przetworzenia: {len(URL_LIST)}")

In [ ]:
# --- KOMÓRKA 4: SILNIK BACKUPU (GŁÓWNY SKRYPT) ---
import time
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, WebDriverException

def get_driver():
    chrome_options = Options()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    chrome_options.add_argument('--disable-gpu')
    chrome_options.add_argument(f'--window-size={VIEWPORT_WIDTH},{VIEWPORT_HEIGHT}')
    chrome_options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    
    # Próba użycia menedżera lub ścieżki systemowej
    try:
        from webdriver_manager.chrome import ChromeDriverManager
        from selenium.webdriver.chrome.service import Service
        service = Service(ChromeDriverManager().install())
        driver = webdriver.Chrome(service=service, options=chrome_options)
    except Exception as e:
        # Fallback do ścieżki apt-get w Colabie
        service = Service('/usr/bin/chromedriver')
        driver = webdriver.Chrome(service=service, options=chrome_options)
        
    return driver

def clean_html(html):
    # Proste czyszczenie skryptów analitycznych
    soup = BeautifulSoup(html, 'lxml')
    for tag in soup(['script', 'style']):
        if tag.get('src') and ('google-analytics' in tag.get('src') or 'doubleclick' in tag.get('src')):
            tag.decompose()
    return str(soup)

def backup_site(url, base_dir):
    print(f"\n🚀 Przetwarzanie: {url}")
    domain = urlparse(url).netloc.replace('.', '_')
    site_dir = os.path.join(base_dir, domain)
    os.makedirs(site_dir, exist_ok=True)
    
    driver = None
    try:
        driver = get_driver()
        driver.get(url)
        
        # Czekanie na load i scrollowanie (lazy load)
        time.sleep(WAIT_TIME)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1)
        driver.execute_script("window.scrollTo(0, 0);")
        
        # 1. Zrzut ekranu
        screenshot_path = os.path.join(site_dir, "screenshot.png")
        driver.save_screenshot(screenshot_path)
        print(f"✅ Zapisano screenshot: {screenshot_path}")
        
        # 2. Pobranie HTML
        html_content = driver.page_source
        if MODE == 'clean':
            html_content = clean_html(html_content)
            
        html_path = os.path.join(site_dir, "index.html")
        with open(html_path, 'w', encoding='utf-8') as f:
            f.write(html_content)
        print(f"✅ Zapisano HTML: {html_path}")
        
        # 3. Proste pobranie assets (CSS/IMG) - wersja uproszczona
        if MODE == 'full':
            soup = BeautifulSoup(html_content, 'lxml')
            assets_dir = os.path.join(site_dir, "assets")
            os.makedirs(assets_dir, exist_ok=True)
            
            count = 0
            for img in soup.find_all('img'):
                src = img.get('src')
                if src and src.startswith('http'):
                    try:
                        r = requests.get(src, timeout=5)
                        if r.status_code == 200:
                            fname = os.path.basename(src.split('?')[0])
                            if not fname: fname = f"img_{count}.jpg"
                            with open(os.path.join(assets_dir, fname), 'wb') as f:
                                f.write(r.content)
                            count += 1
                    except:
                        pass
            print(f"✅ Pobrano {count} obrazków.")
            
    except WebDriverException as e:
        print(f"❌ Błąd WebDriver dla {url}: {e}")
    except Exception as e:
        print(f"❌ Błąd krytyczny dla {url}: {e}")
    finally:
        if driver:
            driver.quit()

# --- URUCHOMIENIE ---
if __name__ == "__main__":
    print(f"Rozpoczynanie backupu w trybie: {MODE}")
    for url in URL_LIST:
        backup_site(url, BACKUP_DIR)
    print("\n🎉 Wszystkie zadania zakończone! Sprawdź folder na Google Drive.")